# Multimodal Acquisition, Calibration, And Manifests

PyTex now has a stable shared experiment layer. This notebook shows how acquisition
semantics, calibration state, quality metadata, and machine-readable manifests fit
together.


In [ ]:
from pathlib import Path
import tempfile

import numpy as np

from pytex import (
    AcquisitionGeometry,
    AtomicSite,
    BenchmarkManifest,
    build_crystal_scene,
    CalibrationRecord,
    CrystalCellOverlay,
    CrystalDirection,
    CrystalDirectionOverlay,
    CrystalMap,
    CrystalPlane,
    CrystalPlaneOverlay,
    DirectionAnnotationStyle,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    list_style_themes,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    PowderPattern,
    PowderReflection,
    ReferenceFrame,
    resolve_style,
    RadiationSpec,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    UnitCell,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
    PlaneAnnotationStyle,
    generate_saed_pattern,
    generate_xrd_pattern,
    plot_odf,
    plot_crystal_structure_3d,
    plot_inverse_pole_figure,
    plot_orientations,
    plot_pole_figure,
    plot_saed_pattern,
    plot_symmetry_elements,
    plot_symmetry_orbit,
    plot_vector_set,
    plot_xrd_pattern,
)


def make_context():
    crystal = ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    symmetry = SymmetrySpec.from_point_group("m-3m", reference_frame=crystal)
    lattice = Lattice(3.6, 3.6, 3.6, 90.0, 90.0, 90.0, crystal_frame=crystal)
    unit_cell = UnitCell(
        lattice=lattice,
        sites=(
            AtomicSite("A1", "Cu", np.array([0.0, 0.0, 0.0])),
            AtomicSite("A2", "Cu", np.array([0.0, 0.5, 0.5])),
            AtomicSite("A3", "Cu", np.array([0.5, 0.0, 0.5])),
            AtomicSite("A4", "Cu", np.array([0.5, 0.5, 0.0])),
        ),
    )
    phase = Phase(
        "fcc_demo",
        lattice=lattice,
        symmetry=symmetry,
        crystal_frame=crystal,
        unit_cell=unit_cell,
    )
    return crystal, specimen, map_frame, detector, lab, phase


def make_nacl_phase():
    crystal = ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )
    lattice = Lattice(5.6402, 5.6402, 5.6402, 90.0, 90.0, 90.0, crystal_frame=crystal)
    symmetry = SymmetrySpec.from_point_group("m-3m", reference_frame=crystal)
    unit_cell = UnitCell(
        lattice=lattice,
        sites=(
            AtomicSite("Cl1", "Cl", np.array([0.0, 0.0, 0.0])),
            AtomicSite("Cl2", "Cl", np.array([0.0, 0.5, 0.5])),
            AtomicSite("Cl3", "Cl", np.array([0.5, 0.0, 0.5])),
            AtomicSite("Cl4", "Cl", np.array([0.5, 0.5, 0.0])),
            AtomicSite("Na1", "Na", np.array([0.5, 0.0, 0.0])),
            AtomicSite("Na2", "Na", np.array([0.0, 0.5, 0.0])),
            AtomicSite("Na3", "Na", np.array([0.0, 0.0, 0.5])),
            AtomicSite("Na4", "Na", np.array([0.5, 0.5, 0.5])),
        ),
    )
    return Phase(
        "NaCl",
        lattice=lattice,
        symmetry=symmetry,
        crystal_frame=crystal,
        unit_cell=unit_cell,
    )


def load_zr_hcp_phase():
    crystal = ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )
    return Phase.from_cif(Path("fixtures/phases/zr_hcp/phase.cif"), crystal_frame=crystal)


def publication_crystal_style():
    return {
        "crystal": {
            "atom_radius_scale": 0.5,
            "atom_edgewidth": 0.0,
            "atom_surface_resolution": 34,
            "bond_surface_resolution": 28,
            "bond_alpha": 0.72,
            "bond_color": "#7c8ea3",
            "atom_specular_strength": 0.42,
            "light_specular": 0.4,
        }
    }


In [ ]:
crystal, specimen, map_frame, detector, lab, phase = make_context()

acquisition = AcquisitionGeometry(
    specimen_frame=specimen,
    modality="ebsd",
    map_frame=map_frame,
    specimen_to_map=FrameTransform(
        source=specimen,
        target=map_frame,
        rotation_matrix=np.eye(3),
    ),
    calibration_record=CalibrationRecord(
        source="stage-fit",
        status="calibrated",
        residual_error=0.1,
    ),
    measurement_quality=MeasurementQuality(
        confidence=0.95,
        valid_fraction=0.99,
        uncertainty={"tilt_deg": 0.1},
    ),
)

experiment = ExperimentManifest.from_acquisition_geometry(
    acquisition,
    source_system="pytex",
    phase=phase,
    referenced_files=("scan.ang",),
)
print(experiment.to_dict()["schema_id"])
print(experiment.to_dict()["modality"])


In [ ]:
benchmark = BenchmarkManifest(
    benchmark_id="ebsd_regular_grid_demo",
    subsystem="ebsd",
    baseline_kind="internal_plus_mtex",
    workflows=("kam", "segmentation", "grod"),
    tolerances={"misorientation_atol_deg": 1e-6},
)

validation = ValidationManifest(
    campaign_name="texture_validation_demo",
    subsystem="texture",
    baseline_kind="mtex_plus_internal",
    status="implemented",
    reference_ids=("MTEX", "Bunge"),
    linked_benchmark_ids=(benchmark.benchmark_id,),
)

workflow = WorkflowResultManifest(
    result_id="demo_result",
    workflow_name="ebsd_pipeline",
    modality="ebsd",
    produced_by="pytex",
    input_manifest_ids=(experiment.schema_id,),
    artifact_paths=("results/demo_figure.svg",),
)

print(benchmark.to_dict()["workflows"])
print(validation.to_dict()["reference_ids"])
print(workflow.to_dict()["artifact_paths"])
